## APIs: collect weather data

Weather is a huge factor when it comes to scooter usage. On very cold days and, especially, on rainy days, hardly anyone wants to use a scooter.

Thankfully, there is an API that provides free weather forecasts: [OpenWeather](https://openweathermap.org/api).

If you followed our guidance, you already have created an account. If you forgot, go back to the LMS section 🔧Setting up accounts for the project.

OpenWeather has many APIs that you can use. Once your API key arrives, we recommend using the [5 day forecast](https://openweathermap.org/forecast5). But if you feel confident and wish to explore, you are of course free to choose another one of the APIs.

It’s easy to make an API call using code similar to below:

```python
f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API key}&units=metric"
```

Obviously, latitude, longitude, and API_key are variables that you will have already defined. Once you’ve made this API call, you’ll be able to explore the received information using the [requests](https://docs.python-requests.org/en/latest/) library and pandas.

### 🧩 The goal for today is to automate the collection of weather data for multiple cities. 

1. Explore the data received from the API. What’s useful? What’s not?
2. Extract the information you see as useful and make a DataFrame from it
3. Consolidate the steps you took to make the DataFrame into a function. The function should output a DataFrame with the weather of multiple cities, when given a list of cities as an input. If you’re still unsure about functions, here’s a guide.
4. You should be able to use this function to get the weather data for the cities you web scraped yesterday



In [2]:
import os 
from dotenv import load_dotenv
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re
import pymysql

load_dotenv()

connection_string = f'mysql+pymysql://{os.environ["MYSQL_USER"]}:{os.environ["MYSQL_ROOT_PASSWORD"]}@{os.environ["MYSQL_HOST"]}:{os.environ["MYSQL_PORT"]}/gans' #/{schema}'
api_key = os.environ["API_KEY_OPENWEATHER"]

In [3]:
def get_weather(lat, lon):
    return requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={api_key}&units=metric")

In [4]:
def city_loc(city_df):
    forecast = {
        "city": [],
        "weather": [],
        "short": [],
    }
    for _, i in city_df.iterrows():
        weather = get_weather(i["latitude"], i["longitude"]).json()
        
        forecast["city"].append(pd.json_normalize(weather["city"]))
        forecast["city"][-1]["city_name"] = i["name"]
        forecast["city"][-1]["city_id"] = i["city_id"]
        forecast["city"][-1]["country_name"] = i["country"]
        
        fc = pd.json_normalize(weather["list"]).drop("weather", axis=1)
        fc["city_id"] = weather["city"]["id"]
        fc["city_id_sql"] = i["city_id"]
        forecast["weather"].append(fc)
    
        short = pd.json_normalize(
            weather["list"],
            record_path="weather",   # explode each row's list of dicts
            meta=["dt"],             # parent fields to attach to every exploded row
        )
        short["city_id"] = weather["city"]["id"]
        forecast["short"].append(short)

    for k, i in forecast.items():
        forecast[k] = pd.concat(i, ignore_index=True)

    return forecast

In [5]:
df = pd.read_sql(
    "SELECT * FROM city",
    con = connection_string,
)

In [6]:
df

,city_id,name,country,country_code,timezone,population,latitude,longitude
0,1,Berlin,Germany,DE,7200,3596999,52.518667,13.403000
1,2,Hamburg,Germany,DE,7200,1973896,53.550000,10.000000
2,3,Munich,Germany,DE,7200,1505036,48.135833,11.571667


In [7]:
forecast = city_loc(df)

In [8]:
forecast["city"]

,id,name,country,population,timezone,sunrise,sunset,coord.lat,coord.lon,city_name,city_id,country_name
0,6545310,Mitte,DE,329078,3600,1793080375,1793116044,52.5187,13.4030,Berlin,1,Germany
1,2911298,Hamburg,DE,1739117,3600,1793081345,1793116707,53.5500,10.0000,Hamburg,2,Germany
2,2867714,Munich,DE,1260391,3600,1793080241,1793117057,48.1358,11.5717,Munich,3,Germany


In [9]:
if not "rain.3h" in forecast["weather"].columns:
    forecast["weather"]["rain.3h"] = 0

In [10]:
forecast["weather"]

,dt,visibility,pop,dt_txt,main.temp,main.feels_like,main.temp_min,main.temp_max,main.pressure,main.sea_level,...,main.temp_kf,main.dew_point,clouds.all,wind.speed,wind.deg,wind.gust,sys.pod,rain.3h,city_id,city_id_sql
0,1793102400,10000,0.0,2026-10-27 12:00:00,14.87,14.37,14.78,14.87,1019,1019,...,0.09,10.48,75,2.95,159,6.97,d,NaN,6545310,1
1,1793113200,10000,0.0,2026-10-27 15:00:00,14.57,13.99,13.96,14.57,1019,1019,...,0.61,9.79,52,3.21,146,6.54,d,NaN,6545310,1
2,1793124000,10000,0.0,2026-10-27 18:00:00,10.69,9.80,8.60,10.69,1019,1019,...,2.09,6.64,28,3.63,154,7.17,n,NaN,6545310,1
3,1793134800,10000,0.0,2026-10-27 21:00:00,8.78,6.94,8.78,8.78,1019,1019,...,0.00,6.01,13,3.18,165,6.17,n,NaN,6545310,1
4,1793145600,10000,0.0,2026-10-28 00:00:00,9.40,8.03,9.40,9.40,1019,1019,...,0.00,5.74,40,2.62,169,4.98,n,NaN,6545310,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,1793480400,10000,0.0,2026-10-31 21:00:00,3.99,3.99,3.99,3.99,1031,1031,...,0.00,2.95,1,0.46,99,0.81,n,NaN,2867714,3
116,1793491200,239,0.0,2026-11-01 00:00:00,2.00,2.00,2.00,2.00,1031,1031,...,0.00,1.86,100,0.55,14,0.76,n,NaN,2867714,3
117,1793502000,116,0.0,2026-11-01 03:00:00,0.66,0.66,0.66,0.66,1031,1031,...,0.00,0.51,100,0.54,195,1.91,n,NaN,2867714,3
118,1793512800,107,0.0,2026-11-01 06:00:00,1.74,1.74,1.74,1.74,1032,1032,...,0.00,1.38,100,0.31,53,1.62,d,NaN,2867714,3


In [11]:
forecast["short"]

,id,main,description,icon,dt,city_id
0,803,Clouds,broken clouds,04d,1793102400,6545310
1,803,Clouds,broken clouds,04d,1793113200,6545310
2,802,Clouds,scattered clouds,03n,1793124000,6545310
3,801,Clouds,few clouds,02n,1793134800,6545310
4,802,Clouds,scattered clouds,03n,1793145600,6545310
...,...,...,...,...,...,...
115,800,Clear,clear sky,01n,1793480400,2867714
116,804,Clouds,overcast clouds,04n,1793491200,2867714
117,804,Clouds,overcast clouds,04n,1793502000,2867714
118,804,Clouds,overcast clouds,04d,1793512800,2867714


### Compose data for database

#### Constant Information

- city_id
- city_name
- country
- country_code
- timezone
- latitude
- longitude
- population

#### Annual Information

Normally the population, but for the purpose of this project, the population is considered constant.

#### Daily Information

- sunrise
- sunset

#### 3h Information

- dt
- dt_txt
- visibility
- probability of precipitation (pop)
- main.temp
- main.feels_like
- main.pressure
- main.humidity
- wind.speed
- wind.deg
- wind.gust
- sys.pod (part of day)
- rain.3h
- weather_id
- weather.main
- description
- icon
- dt
- city_id

```sql
CREATE TABLE city (
    city_id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(255) NOT NULL,
    country VARCHAR(255) NOT NULL,
    country_code CHAR(2),
    timezone INT,
    population INT,
    latitude DECIMAL(10 , 7 ),
    longitude DECIMAL(10 , 7 )
);

CREATE TABLE solar_time (
    city_id INT NOT NULL,
    sunrise INT NOT NULL,
    sunset INT NOT NULL,
    sunrise_text TEXT,
    sunset_text TEXT,
    FOREIGN KEY (city_id)
        REFERENCES city (city_id),
	UNIQUE (city_id, sunrise)
);

CREATE TABLE forecast (
	city_id INT NOT NULL,
    `time` INT NOT NULL,
    time_text TEXT,
    temperature DECIMAL(5, 2),
    temperature_felt DECIMAL(5, 2),
    weather TEXT,
    `description` TEXT,
    weather_id TEXT,
    precipitation_prob INT,
    rain_3h DECIMAL(5, 2),
    clouds INT,
    pressure INT,
    humidity INT,
    icon_id CHAR(3),
    FOREIGN KEY (city_id)
        REFERENCES city (city_id),
	UNIQUE (city_id, `time`)
);
```

In [12]:
forecast["city"]

,id,name,country,population,timezone,sunrise,sunset,coord.lat,coord.lon,city_name,city_id,country_name
0,6545310,Mitte,DE,329078,3600,1793080375,1793116044,52.5187,13.4030,Berlin,1,Germany
1,2911298,Hamburg,DE,1739117,3600,1793081345,1793116707,53.5500,10.0000,Hamburg,2,Germany
2,2867714,Munich,DE,1260391,3600,1793080241,1793117057,48.1358,11.5717,Munich,3,Germany


In [13]:
city_update = forecast["city"].loc[:, ["city_id", "country", "timezone"]]

In [14]:
city_update = city_update.rename({"country": "country_code"}, axis=1)

In [15]:
solar_time = forecast["city"].loc[:, ["city_id", "sunrise", "sunset"]]

In [16]:
solar_time["sunrise_text"] = pd.to_datetime(solar_time.loc[:, "sunrise"] + forecast["city"].loc[:, "timezone"], unit="s").dt.strftime("%Y-%m-%d %H-%M-%S")
solar_time["sunset_text"] = pd.to_datetime(solar_time.loc[:, "sunset"] + forecast["city"].loc[:, "timezone"], unit="s").dt.strftime("%Y-%m-%d %H-%M-%S")

In [17]:
solar_time

,city_id,sunrise,sunset,sunrise_text,sunset_text
0,1,1793080375,1793116044,2026-10-27 06-52-55,2026-10-27 16-47-24
1,2,1793081345,1793116707,2026-10-27 07-09-05,2026-10-27 16-58-27
2,3,1793080241,1793117057,2026-10-27 06-50-41,2026-10-27 17-04-17


In [18]:
forecast["weather"]

,dt,visibility,pop,dt_txt,main.temp,main.feels_like,main.temp_min,main.temp_max,main.pressure,main.sea_level,...,main.temp_kf,main.dew_point,clouds.all,wind.speed,wind.deg,wind.gust,sys.pod,rain.3h,city_id,city_id_sql
0,1793102400,10000,0.0,2026-10-27 12:00:00,14.87,14.37,14.78,14.87,1019,1019,...,0.09,10.48,75,2.95,159,6.97,d,NaN,6545310,1
1,1793113200,10000,0.0,2026-10-27 15:00:00,14.57,13.99,13.96,14.57,1019,1019,...,0.61,9.79,52,3.21,146,6.54,d,NaN,6545310,1
2,1793124000,10000,0.0,2026-10-27 18:00:00,10.69,9.80,8.60,10.69,1019,1019,...,2.09,6.64,28,3.63,154,7.17,n,NaN,6545310,1
3,1793134800,10000,0.0,2026-10-27 21:00:00,8.78,6.94,8.78,8.78,1019,1019,...,0.00,6.01,13,3.18,165,6.17,n,NaN,6545310,1
4,1793145600,10000,0.0,2026-10-28 00:00:00,9.40,8.03,9.40,9.40,1019,1019,...,0.00,5.74,40,2.62,169,4.98,n,NaN,6545310,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,1793480400,10000,0.0,2026-10-31 21:00:00,3.99,3.99,3.99,3.99,1031,1031,...,0.00,2.95,1,0.46,99,0.81,n,NaN,2867714,3
116,1793491200,239,0.0,2026-11-01 00:00:00,2.00,2.00,2.00,2.00,1031,1031,...,0.00,1.86,100,0.55,14,0.76,n,NaN,2867714,3
117,1793502000,116,0.0,2026-11-01 03:00:00,0.66,0.66,0.66,0.66,1031,1031,...,0.00,0.51,100,0.54,195,1.91,n,NaN,2867714,3
118,1793512800,107,0.0,2026-11-01 06:00:00,1.74,1.74,1.74,1.74,1032,1032,...,0.00,1.38,100,0.31,53,1.62,d,NaN,2867714,3


In [19]:
forecast_table = (
    forecast["weather"]
        .set_index(["dt", "city_id"])
        .join(
            forecast["short"].set_index(["dt", "city_id"]),
            how="inner"
        )
        .reset_index()
        .drop_duplicates(["dt", "city_id"])
        .loc[:, [
            "city_id_sql",
            "dt",
            "dt_txt",
            "main.temp",
            "main.feels_like",
            "main",
            "description",
            "id",
            "rain.3h",
            "clouds.all",
            "main.pressure",
            "main.humidity",
            "icon",
        ]]
        .rename({
            "dt": "time",
            "dt_txt": "time_text",
            "main.temp": "temperature",
            "main.feels_like": "temperature_felt",
            "main": "weather",
            "id": "weather_id",
            "rain.3h": "rain_3h",
            "clouds.all": "clouds",
            "main.pressure": "pressure",
            "main.humidity": "humidity",
            "icon": "icon_id",
            "city_id_sql": "city_id",
        }, axis=1)
)
forecast_table["precipitation_prob"] = (forecast["weather"].loc[:, "pop"] * 100).astype(int)

In [26]:
", ".join(solar_time.columns.values)

'city_id, sunrise, sunset, sunrise_text, sunset_text'

In [25]:
s = ""
for i in solar_time.columns.values:
    s += f"{i} = t.{i}" + ",\n"
print(s)

city_id = t.city_id,
sunrise = t.sunrise,
sunset = t.sunset,
sunrise_text = t.sunrise_text,
sunset_text = t.sunset_text,



In [31]:
city_update.to_sql(
    'city_tmp',
    schema="gans",
    if_exists='replace',
    con=connection_string,
    index=False,
)

3

In [32]:
solar_time.to_sql(
    'solar_time_tmp',
    schema="gans",
    if_exists='replace',
    con=connection_string,
    index=False,
)

3

In [33]:
forecast_table.to_sql(
    'forecast_tmp',
    schema="gans",
    if_exists='replace',
    con=connection_string,
    index=False,
)

120

In [34]:
conn = pymysql.connect(
    host=os.environ["MYSQL_HOST"],
    port=int(os.environ["MYSQL_PORT"]),
    user=os.environ["MYSQL_USER"],
    password=os.environ["MYSQL_ROOT_PASSWORD"],
    database="gans",
)

with conn.cursor() as c:
    c.callproc("update_city")
    c.callproc("update_solar_time")
    c.callproc("update_forecast")

conn.commit()
conn.close()